# Experiment 4: Basic Disease Risk Prediction and Performance Evaluation Using Regression ML Models
**AI & ML in Healthcare Lab (HAIMLSBL701)**

**Aim:** To perform basic risk prediction and performance evaluation using Linear, Multivariate and Logistic Regression ML models.

**Software Tools:** Jupyter Notebook and Python 3.x

**Dataset:** UCI Machine Learning Repository (loaded via `ucimlrepo`, `id=857`)

In [ ]:
# Run once if the package is not installed
# !pip install ucimlrepo scikit-learn pandas numpy matplotlib

## Part 1: Linear Regression
Simple Linear Regression uses one independent variable to predict a continuous dependent variable:

**Y = a + bX**, where *a* is the intercept and *b* is the slope.

Here, **Hemoglobin** is used to predict **Packed Cell Volume**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
ckd = fetch_ucirepo(id=857)
X = ckd.data.features
y = ckd.data.targets
data = pd.concat([X, y], axis=1)

print("Dataset Shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())

In [ ]:
# Convert range values into numerical values
def convert_value(value):
    value = str(value).strip()
    # Remove symbols
    value = value.replace("≥", "").replace("<", "").strip()
    # Handle ranges
    if "-" in value:
        parts = value.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    try:
        return float(value)
    except:
        return np.nan

# Convert Hemoglobin and Packed Cell Volume
data["Hemoglobin"] = data["hemo"].apply(convert_value)
data["PackedCellVolume"] = data["pcv"].apply(convert_value)

# Remove missing values
data = data.dropna(subset=["Hemoglobin", "PackedCellVolume"])

# Input and target
X = data[["Hemoglobin"]]
y = data["PackedCellVolume"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Create and train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("========== Linear Regression ==========")
print("Model trained successfully.")
print("Slope (Coefficient):", model.coef_[0])
print("Intercept:", model.intercept_)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nMean Squared Error =", round(mse, 4))
print("R² Score =", round(r2, 4))
print("R² Score (%) =", round(r2 * 100, 2), "%")

In [ ]:
# Sort values for regression line
sort_index = np.argsort(X_test["Hemoglobin"].values)
X_test_sorted = X_test["Hemoglobin"].values[sort_index]
y_pred_sorted = y_pred[sort_index]

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(X_test["Hemoglobin"], y_test, alpha=0.6, label="Actual Data")
plt.plot(X_test_sorted, y_pred_sorted, linewidth=2, label="Linear Regression Line")
plt.xlabel("Hemoglobin")
plt.ylabel("Packed Cell Volume")
plt.title("Linear Regression - Hemoglobin vs Packed Cell Volume")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: Multivariate Linear Regression
Multivariate Linear Regression uses **multiple independent variables** to predict one continuous target:

**Y = b₀ + b₁X₁ + b₂X₂ + ... + bₙXₙ**

Here, Hemoglobin, Blood Glucose, Blood Urea, Sodium, Potassium, Serum Creatinine and Age are used to predict **Packed Cell Volume**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load CKD dataset
ckd = fetch_ucirepo(id=857)
X_data = ckd.data.features
y_data = ckd.data.targets
data = pd.concat([X_data, y_data], axis=1)

print("Dataset Shape:", data.shape)
print("\nFirst 5 Rows:")
print(data.head())

In [ ]:
# Function to convert range/category values into numbers
def convert_value(value):
    value = str(value).strip()
    value = value.replace("≥", "").replace("<", "").strip()
    if "-" in value:
        parts = value.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    try:
        return float(value)
    except:
        return np.nan

# Convert required attributes
data["Hemoglobin"] = data["hemo"].apply(convert_value)
data["PackedCellVolume"] = data["pcv"].apply(convert_value)
data["BloodGlucose"] = data["bgr"].apply(convert_value)
data["BloodUrea"] = data["bu"].apply(convert_value)
data["Sodium"] = data["sod"].apply(convert_value)
data["Potassium"] = data["pot"].apply(convert_value)
data["SerumCreatinine"] = data["sc"].apply(convert_value)
data["Age"] = data["age"].apply(convert_value)

feature_cols = [
    "Hemoglobin", "BloodGlucose", "BloodUrea",
    "Sodium", "Potassium", "SerumCreatinine", "Age"
]

# Select input features and target
X = data[feature_cols]
y = data["PackedCellVolume"]

# Remove missing values
valid_data = pd.concat([X, y], axis=1).dropna()
X = valid_data[feature_cols]
y = valid_data["PackedCellVolume"]

print("\nValid Dataset Shape:", valid_data.shape)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Create and train model
model = LinearRegression()
model.fit(X_train, y_train)

print("========== Multivariate Linear Regression ==========")
print("Model trained successfully.")

# Display coefficients
print("\nModel Coefficients:")
for feature, coefficient in zip(X.columns, model.coef_):
    print(feature, ":", coefficient)
print("\nIntercept:", model.intercept_)

# Predictions
y_pred = model.predict(X_test)
print("\nFirst 10 Predictions:")
print(y_pred[:10])

# Model evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nMean Squared Error =", round(mse, 4))
print("R² Score =", round(r2, 4))
print("R² Score (%) =", round(r2 * 100, 2), "%")

In [ ]:
# Actual vs Predicted graph
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, label="Predicted Values")

# Perfect prediction line
minimum = min(y_test.min(), y_pred.min())
maximum = max(y_test.max(), y_pred.max())
plt.plot([minimum, maximum], [minimum, maximum], linewidth=2, label="Perfect Prediction")

plt.xlabel("Actual Packed Cell Volume")
plt.ylabel("Predicted Packed Cell Volume")
plt.title("Actual vs Predicted - Multivariate Linear Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Logistic Regression
Logistic Regression is a supervised **classification** algorithm. It uses the sigmoid function to convert the model output into a probability:

**σ(z) = 1 / (1 + e⁻ᶻ)**

If the probability is ≥ 0.5, the sample is classified as class 1 (**CKD**); otherwise class 0 (**No CKD**).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load CKD dataset
ckd = fetch_ucirepo(id=857)
X_data = ckd.data.features
y_data = ckd.data.targets
data = pd.concat([X_data, y_data], axis=1)

print("Dataset Shape:", data.shape)
print("\nFirst 5 Rows:")
print(data.head())

In [ ]:
# Convert range/category values into numerical values
def convert_value(value):
    value = str(value).strip()
    value = value.replace("≥", "").replace("<", "").strip()
    if "-" in value:
        parts = value.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    try:
        return float(value)
    except:
        return np.nan

# Convert required attributes
data["Hemoglobin"] = data["hemo"].apply(convert_value)
data["PackedCellVolume"] = data["pcv"].apply(convert_value)
data["BloodGlucose"] = data["bgr"].apply(convert_value)
data["BloodUrea"] = data["bu"].apply(convert_value)
data["Sodium"] = data["sod"].apply(convert_value)
data["Potassium"] = data["pot"].apply(convert_value)
data["SerumCreatinine"] = data["sc"].apply(convert_value)
data["Age"] = data["age"].apply(convert_value)

# Convert target variable
data["CKD"] = data["class"].astype(str).str.lower().map({"ckd": 1, "notckd": 0})

# Select multiple input features
features = [
    "Hemoglobin", "PackedCellVolume", "BloodGlucose", "BloodUrea",
    "Sodium", "Potassium", "SerumCreatinine", "Age"
]

X = data[features]
y = data["CKD"]

# Remove missing values
valid_data = pd.concat([X, y], axis=1).dropna()
X = valid_data[features]
y = valid_data["CKD"]

print("\nValid Dataset Shape:", valid_data.shape)
print("\nClass Distribution:")
print(y.value_counts())

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Create and train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("========== Logistic Regression ==========")
print("Model trained successfully.")

# Display coefficients
print("\nModel Coefficients:")
for feature, coefficient in zip(features, model.coef_[0]):
    print(feature, ":", coefficient)
print("\nIntercept:", model.intercept_[0])

# Predictions
y_pred = model.predict(X_test_scaled)

# Prediction probabilities
y_probability = model.predict_proba(X_test_scaled)[:, 1]

print("\nFirst 10 Predictions:")
print(y_pred[:10])
print("\nFirst 10 Prediction Probabilities:")
print(y_probability[:10])

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy =", round(accuracy * 100, 2), "%")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No CKD", "CKD"], zero_division=0))

In [ ]:
# Confusion Matrix Visualization
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.colorbar()
plt.xticks([0, 1], ["No CKD", "CKD"])
plt.yticks([0, 1], ["No CKD", "CKD"])
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black", fontsize=14)

plt.tight_layout()
plt.show()

### Sigmoid curve: Hemoglobin vs probability of CKD

In [ ]:
sigmoid_data = data[["Hemoglobin", "CKD"]].dropna()
X_sigmoid = sigmoid_data[["Hemoglobin"]]
y_sigmoid = sigmoid_data["CKD"]

# Split data for sigmoid model
X_train_sig, X_test_sig, y_train_sig, y_test_sig = train_test_split(
    X_sigmoid, y_sigmoid, test_size=0.20, random_state=42, stratify=y_sigmoid
)

# Scale Hemoglobin
scaler_sig = StandardScaler()
X_train_sig_scaled = scaler_sig.fit_transform(X_train_sig)
X_test_sig_scaled = scaler_sig.transform(X_test_sig)

# Single-variable Logistic Regression
logistic_sigmoid = LogisticRegression(max_iter=1000, random_state=42)
logistic_sigmoid.fit(X_train_sig_scaled, y_train_sig)

# Smooth range of Hemoglobin values
x_values = np.linspace(
    X_sigmoid["Hemoglobin"].min(), X_sigmoid["Hemoglobin"].max(), 200
).reshape(-1, 1)

# Probability of CKD
x_scaled = scaler_sig.transform(x_values)
probability = logistic_sigmoid.predict_proba(x_scaled)[:, 1]

# Plot Sigmoid Curve
plt.figure(figsize=(8, 6))
plt.scatter(X_test_sig["Hemoglobin"], y_test_sig, alpha=0.6, label="Actual Data")
plt.plot(x_values, probability, linewidth=2, label="Sigmoid Curve")
plt.axhline(0.5, linestyle="--", linewidth=1.5, label="Decision Threshold = 0.5")
plt.xlabel("Hemoglobin")
plt.ylabel("Probability of CKD")
plt.title("Logistic Regression - Hemoglobin vs CKD Probability")
plt.yticks([0, 0.5, 1], ["No CKD", "Threshold", "CKD"])
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Sigmoid model accuracy
sigmoid_predictions = logistic_sigmoid.predict(X_test_sig_scaled)
sigmoid_accuracy = accuracy_score(y_test_sig, sigmoid_predictions)

print("\n========== Sigmoid Model ==========")
print("Hemoglobin-based Logistic Regression Accuracy =", round(sigmoid_accuracy * 100, 2), "%")

## Conclusion
The experiment demonstrated Linear Regression, Multivariate Linear Regression and Logistic Regression in Python. Linear and Multivariate Linear Regression predict continuous values, whereas Logistic Regression is used for classification. This gave a practical understanding of these supervised learning techniques and their applications in healthcare.